# CSI Training on Kaggle with Git Clone
Notebook nay dung workflow: clone repo -> checkout tag/branch -> install -> train -> save outputs.

## 1) Kaggle settings
Bat GPU + Internet trong Settings truoc khi chay.
Neu data khong nam trong repo, Add Data (Kaggle Dataset) truoc.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('GPU not available. Check Kaggle Settings.')

In [ ]:
# TODO: doi REPO_URL theo repo cua ban
REPO_URL = 'https://github.com/your-user/PBL5-train-model.git'
BRANCH = 'feat/train_ai'
TAG = 'v1.0'  # de rong neu khong muon checkout tag
PROJECT_DIR = '/kaggle/working/PBL5-train-model'

!rm -rf {PROJECT_DIR}
!git clone -b {BRANCH} {REPO_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}

if TAG:
    !git checkout {TAG}

!git rev-parse --short HEAD
!git log --oneline -n 3

In [ ]:
# Optional: copy data tu Kaggle Dataset neu data khong nam trong repo
# DATASET_DIR = '/kaggle/input/pbl5-v1-data'
# !cp {DATASET_DIR}/sit.csv data/raw/sit.csv
# !cp {DATASET_DIR}/stand.csv data/raw/stand.csv

!python -m pip install -q -r requirements.txt

import os
required_files = [
    'data/raw/sit.csv',
    'data/raw/stand.csv',
    'configs/train_default.json',
]
missing = [p for p in required_files if not os.path.exists(p)]
print('Missing files:', missing)
assert not missing, f'Missing files: {missing}'
print('Setup OK')

In [ ]:
# Train CNN2D
!python -m src.train \
  --config configs/train_default.json \
  --model-type cnn2d \
  --epochs 20 \
  --batch-size 16 \
  --run-name kaggle_git_cnn2d \
  --output-dir /kaggle/working/experiments

In [ ]:
# Optional: Train LSTM-CNN
# !python -m src.train \
#   --config configs/train_default.json \
#   --model-type lstmcnn \
#   --epochs 30 \
#   --batch-size 16 \
#   --run-name kaggle_git_lstmcnn \
#   --output-dir /kaggle/working/experiments

In [ ]:
import os, shutil

print('Checkpoints:')
!ls -lah /kaggle/working/models/checkpoints
print('Results:')
!ls -lah /kaggle/working/experiments

os.makedirs('/kaggle/working/output', exist_ok=True)
shutil.make_archive('/kaggle/working/output/checkpoints', 'zip', '/kaggle/working/models/checkpoints')
shutil.make_archive('/kaggle/working/output/experiments', 'zip', '/kaggle/working/experiments')
print('Saved artifacts in /kaggle/working/output')
print(os.listdir('/kaggle/working/output'))